# Calculating drawdown values for refined trading strategy

In [44]:
# Basic libraries 
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pykalman import KalmanFilter
from itertools import combinations
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.stattools import coint
import statsmodels.api as sm
import pickle
import os
import sys

PROJECT_ROOT = os.path.abspath("..")
sys.path.insert(0, PROJECT_ROOT)

In [45]:
# Scripts 
from src.local_config import *
from src.drawdown_optimisation import * 
from src.volatility_plot import *

In [46]:
# Load dictionaries from previous notebook
# In-Sample
with open(PROJECT_ROOT / "data/dictionaries/static_spreads_is.pkl", "rb") as f:
    static_spreads_is = pickle.load(f)
with open(PROJECT_ROOT / "data/dictionaries/static_models_is.pkl", "rb") as f:
    static_models_is = pickle.load(f)

with open(PROJECT_ROOT / "data/dictionaries/dynamic_spreads_is.pkl", "rb") as f:
    dynamic_spreads_is = pickle.load(f)
with open(PROJECT_ROOT / "data/dictionaries/dynamic_details_is.pkl", "rb") as f:
    dynamic_details_is = pickle.load(f)

# Out-Of-Sample
with open(PROJECT_ROOT / "data/dictionaries/static_spreads_oos.pkl", "rb") as f:
    static_spreads_oos = pickle.load(f)
with open(PROJECT_ROOT / "data/dictionaries/static_models_oos.pkl", "rb") as f:
    static_models_oos = pickle.load(f)

with open(PROJECT_ROOT / "data/dictionaries/dynamic_spreads_oos.pkl", "rb") as f:
    dynamic_spreads_oos = pickle.load(f)
with open(PROJECT_ROOT / "data/dictionaries/dynamic_details_oos.pkl", "rb") as f:
    dynamic_details_oos = pickle.load(f)


In [47]:
# Load pairs
with open(PROJECT_ROOT / "data/pairs_is.pkl", "rb") as f:
    pairs_is = pickle.load(f)
with open(PROJECT_ROOT / "data/pairs_oos.pkl", "rb") as f:
    pairs_oss = pickle.load(f)

In [48]:
# Trading Signals
with open(PROJECT_ROOT / "data/dictionaries/is_results.pkl", "rb") as f:
    is_results = pickle.load(f)
with open(PROJECT_ROOT / "data/dictionaries/is_trading_signals.pkl", "rb") as f:
    is_trading_signals = pickle.load(f)

with open(PROJECT_ROOT / "data/dictionaries/oos_results.pkl", "rb") as f:
    oos_results = pickle.load(f)
with open(PROJECT_ROOT / "data/dictionaries/oos_trading_signals.pkl", "rb") as f:
    oos_trading_signals = pickle.load(f)

## Estimated Volatility Formula
Calculated in calculate_vol function:
$$
\hat{\sigma_t}=\sqrt{\frac{\sum_{i=0}^tw_i(r_{t-i}-\mu_t)^2}{\sum_{i=0}^tw_i}}
$$

## Drawdown Threshold Formula
Calculated in drawdown_thresholds function:
$$
\text{Drawdown}=k_{m}\times\sigma_{i,t}\times\sqrt{H}
$$

Where:
- m is either the volatility multiplier coefficient for **reducing** position size or **closing** a position
- $\sigma_{i,t}$ is the estimated volatility from above for pair i
- H is the holding period (which we have fixed to 10 days for simplicity) 

## In-Sample

In [49]:
is_results

{'entry_1.5_exit_0.0': {('700 HK Equity vs 1347 HK Equity',
   0.5):                    y         x   alpha_t    beta_t  spread_t  obs_cov  \
  Date                                                                    
  2015-03-30  4.846201  2.153853  1.155692  1.735673 -0.047875      0.5   
  2015-03-31  4.863326  2.154897  1.157182  1.734420 -0.031353      0.5   
  2015-04-01  4.868749  2.185602  1.162357  1.730626 -0.076066      0.5   
  2015-04-02  4.880838  2.193663  1.167672  1.726793 -0.074835      0.5   
  2015-04-08  4.911728  2.287167  1.194148  1.710678 -0.195027      0.5   
  ...              ...       ...       ...       ...       ...      ...   
  2020-12-24  6.207606  3.808062  4.174089  0.568151 -0.130036      0.5   
  2020-12-28  6.138742  3.793667  4.176448  0.564741 -0.180147      0.5   
  2020-12-29  6.160659  3.816833  4.178816  0.561604 -0.161706      0.5   
  2020-12-30  6.213882  3.820105  4.180271  0.559606 -0.104144      0.5   
  2020-12-31  6.221892  3.775653 

In [50]:
is_extraction = extractor(is_results)

In [51]:
is_extraction.head()

,date,strategy,pair,obs_cov,spread_t,position,spread_change,strategy_ret
1,2015-03-31,entry_1.5_exit_0.0,700 HK Equity vs 1347 HK Equity,0.5,-0.031353,1.0,0.016522,0.016522
2,2015-04-01,entry_1.5_exit_0.0,700 HK Equity vs 1347 HK Equity,0.5,-0.076066,1.0,-0.044714,-0.044714
3,2015-04-02,entry_1.5_exit_0.0,700 HK Equity vs 1347 HK Equity,0.5,-0.074835,1.0,0.001232,0.001232
4,2015-04-08,entry_1.5_exit_0.0,700 HK Equity vs 1347 HK Equity,0.5,-0.195027,1.0,-0.120192,-0.120192
5,2015-04-09,entry_1.5_exit_0.0,700 HK Equity vs 1347 HK Equity,0.5,-0.135044,1.0,0.059984,0.059984


In [52]:
is_volatility = pair_volatility(is_extraction)

In [53]:
is_volatility.head()

,date,strategy,pair,obs_cov,spread_t,position,spread_change,strategy_ret,pair_strategy_vol
3,2015-04-02,entry_1.5_exit_0.0,700 HK Equity vs 1347 HK Equity,0.5,-0.074835,1.0,0.001232,0.001232,0.043300
4,2015-04-08,entry_1.5_exit_0.0,700 HK Equity vs 1347 HK Equity,0.5,-0.195027,1.0,-0.120192,-0.120192,0.031345
5,2015-04-09,entry_1.5_exit_0.0,700 HK Equity vs 1347 HK Equity,0.5,-0.135044,1.0,0.059984,0.059984,0.062142
6,2015-04-10,entry_1.5_exit_0.0,700 HK Equity vs 1347 HK Equity,0.5,-0.180859,1.0,-0.045815,-0.045815,0.056931
7,2015-04-13,entry_1.5_exit_0.0,700 HK Equity vs 1347 HK Equity,0.5,-0.149536,1.0,0.031323,0.031323,0.054406


In [54]:
# Volatility plots
is_volatility_plots = pair_volatility_plots(
    vol_df = is_volatility,
    output_dir=PROJECT_ROOT / "outputs/volatility/in_sample")

Volatility plots saved to: /Users/ivanhung/Documents/GitHub/applied-project-06039211/outputs/volatility/in_sample


In [55]:
is_drawdown_values = drawdown_thresholds(is_volatility)

In [56]:
is_drawdown_values.head()

,date,strategy,pair,obs_cov,pair_strategy_vol,reduce_drawdown_threshold,close_drawdown_threshold
3,2015-04-02,entry_1.5_exit_0.0,700 HK Equity vs 1347 HK Equity,0.5,0.043300,0.684640,1.369280
4,2015-04-08,entry_1.5_exit_0.0,700 HK Equity vs 1347 HK Equity,0.5,0.031345,0.495612,0.991223
5,2015-04-09,entry_1.5_exit_0.0,700 HK Equity vs 1347 HK Equity,0.5,0.062142,0.982552,1.965103
6,2015-04-10,entry_1.5_exit_0.0,700 HK Equity vs 1347 HK Equity,0.5,0.056931,0.900157,1.800313
7,2015-04-13,entry_1.5_exit_0.0,700 HK Equity vs 1347 HK Equity,0.5,0.054406,0.860236,1.720472


### Out-Of-Sample

In [57]:
oos_extraction = extractor(oos_results)

In [58]:
oos_extraction.head()

,date,strategy,pair,obs_cov,spread_t,position,spread_change,strategy_ret
1,2021-03-31,entry_1.5_exit_0.0,700 HK Equity vs 1347 HK Equity,0.5,0.030023,0.0,0.032115,0.000000
2,2021-04-01,entry_1.5_exit_0.0,700 HK Equity vs 1347 HK Equity,0.5,-0.021998,0.0,-0.052021,-0.000000
3,2021-04-07,entry_1.5_exit_0.0,700 HK Equity vs 1347 HK Equity,0.5,-0.128066,0.0,-0.106068,-0.000000
4,2021-04-08,entry_1.5_exit_0.0,700 HK Equity vs 1347 HK Equity,0.5,-0.207898,1.0,-0.079833,-0.000000
5,2021-04-09,entry_1.5_exit_0.0,700 HK Equity vs 1347 HK Equity,0.5,-0.190909,1.0,0.016989,0.016989


In [59]:
oos_volatility = pair_volatility(oos_extraction)

In [60]:
oos_volatility.head()

,date,strategy,pair,obs_cov,spread_t,position,spread_change,strategy_ret,pair_strategy_vol
3,2021-04-07,entry_1.5_exit_0.0,700 HK Equity vs 1347 HK Equity,0.5,-0.128066,0.0,-0.106068,-0.000000,0.000000
4,2021-04-08,entry_1.5_exit_0.0,700 HK Equity vs 1347 HK Equity,0.5,-0.207898,1.0,-0.079833,-0.000000,0.000000
5,2021-04-09,entry_1.5_exit_0.0,700 HK Equity vs 1347 HK Equity,0.5,-0.190909,1.0,0.016989,0.016989,0.000000
6,2021-04-12,entry_1.5_exit_0.0,700 HK Equity vs 1347 HK Equity,0.5,-0.150558,1.0,0.040351,0.040351,0.006306
7,2021-04-13,entry_1.5_exit_0.0,700 HK Equity vs 1347 HK Equity,0.5,-0.130813,1.0,0.019746,0.019746,0.014557


In [61]:
# Volatility plots
oos_volatility_plots = pair_volatility_plots(
    vol_df = oos_volatility,
    output_dir=PROJECT_ROOT / "outputs/volatility/out_of_sample")

Volatility plots saved to: /Users/ivanhung/Documents/GitHub/applied-project-06039211/outputs/volatility/out_of_sample


In [62]:
oos_drawdown_values = drawdown_thresholds(oos_volatility)

In [63]:
oos_drawdown_values.head()

,date,strategy,pair,obs_cov,pair_strategy_vol,reduce_drawdown_threshold,close_drawdown_threshold
3,2021-04-07,entry_1.5_exit_0.0,700 HK Equity vs 1347 HK Equity,0.5,0.000000,0.000000,0.000000
4,2021-04-08,entry_1.5_exit_0.0,700 HK Equity vs 1347 HK Equity,0.5,0.000000,0.000000,0.000000
5,2021-04-09,entry_1.5_exit_0.0,700 HK Equity vs 1347 HK Equity,0.5,0.000000,0.000000,0.000000
6,2021-04-12,entry_1.5_exit_0.0,700 HK Equity vs 1347 HK Equity,0.5,0.006306,0.099702,0.199404
7,2021-04-13,entry_1.5_exit_0.0,700 HK Equity vs 1347 HK Equity,0.5,0.014557,0.230164,0.460327


In [64]:
# Save drawdown_values to .pkl 
with open(PROJECT_ROOT / "data/dictionaries/is_drawdown_values.pkl", "wb") as f:
    pickle.dump(is_drawdown_values, f)

with open(PROJECT_ROOT / "data/dictionaries/oos_drawdown_values.pkl", "wb") as f:
    pickle.dump(oos_drawdown_values, f)